# 03 — Training (OFFLINE, RTX PRO 6000 Blackwell, NO INTERNET)

| attach as input | holds | produces |
|---|---|---|
| the repo dataset | `src/`, `scripts/`, `configs/` | `behaviorsense-runs` |
| the wheels+weights dataset(s) | `*.whl` and `rtmo-l.onnx` | (create/update from `/kaggle/working/runs`) |
| the ADL shards | `*.npz` from notebook 01 | |
| the fall shards | `*.npz` from notebook 02 | |
| `behaviorsense-runs` | previous `last.pt` — **resume runs only** | |

**Dataset names and groupings do not matter.** Notebook 00 emits `wheels/` and `weights/`
as two folders, and publishing them as ONE dataset (e.g. `behavioursense-WW`) or two is
your choice — the resolver below finds each by CONTENT, so both layouts work and the
spelling `behaviour`/`behavior` is tolerated. Attach whatever you have; the cell prints
what it resolved.

Session discipline, in order: **preflight (1 min) → carry forward checkpoints → train**.
Every failure mode this ordering prevents costs hours: wrong-arch torch fails at minute
one instead of minute forty; a resumed run continues bit-exactly instead of restarting.

Resume is bit-exact (RNG state is checkpointed) and refuses changed hyperparameters —
if a cell errors with `resume mismatch`, re-run with the ORIGINAL values rather than
deleting the guard.

In [ ]:
# Resolve every attached asset by CONTENT, not by dataset name.
#
# Kaggle mount paths are not predictable from here, and three separate things vary:
#   - notebook 00 emits two folders, which can be published as ONE dataset or two
#     (observed: a single "behavioursense-WW" holding both wheels/ and weights/)
#   - the dataset title is free text, and "behaviour" vs "behavior" both occur
#   - Save Version nests the working directory inside the dataset, so files end up at
#     <mount>/kaggle/working/... rather than <mount>/...
#
# Guessing the name has already cost one session, and notebook 01's carry-forward bug
# showed how the failure presents: a wrong path reads as "nothing attached", the run
# continues, and work is skipped or destroyed rather than failing loudly.
#
# So identify each asset by a file only it has. A directory holding *.whl is the wheel
# cache no matter what the dataset is called.
import pathlib
from itertools import islice

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    # islice, NOT sorted(...)[:6]. `sorted()` materialises the whole listing
                    # first, and on Kaggle's FUSE mount a slug whose files sit at its root -
                    # `toyota-smarthome-skeleton-v1-2` holds 16,115 - makes that a full
                    # network directory read per mount. Eleven mounts of that shape is most
                    # of the 14 minutes this cell took on the first Toyota run. Six names
                    # are all this diagnostic needs, so stop after six.
                    top = sorted(islice((q.name for q in ds.iterdir()), 6)) \
                        if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []

# MOUNT-INDEXED SEARCH, and why two earlier fixes were not enough.
#
# `INPUT.glob("**/x")` walks every directory under /kaggle/input - 93 minutes with the
# Toyota corpus mounted, notebook 06 measured, and 349 s for the single
# `**/src/behaviorsense/__init__.py` probe in notebook 07's second run. The first "fix",
# FIXED-DEPTH globs like `datasets/*/*/*/rtmo-l.onnx`, was depth-bounded but not
# COST-bounded: to match at depth 3 pathlib scandirs EVERY slug child directory, including
# `toyota-smarthome-skeleton-v1-2`'s 16,115-file root and MSMT17's 65,242 crops.
#
# The mounts are KNOWN at depth 2 (`datasets/<owner>/<slug>/`), so enumerate them once and
# resolve everything else with is_file() stats - one metadata call per mount per candidate,
# never a sibling-directory listing.
_SLUGS = (sorted((INPUT / "datasets").glob("*/*"))
          + sorted((INPUT / "competitions").glob("*")))

def find_fast(tail, what, required=True):
    # Known staging prefixes, each costing one stat per mount:
    #   ""                      files at the slug root
    #   EmotionSense-Extended/  the code dataset was created by zipping the repo FOLDER,
    #                           so everything sits one level below the slug
    #   kaggle/working/         Save Version nests the working directory
    # The prefixes apply to MULTI-COMPONENT tails too. They used to be tried only for bare
    # filenames, which quietly sent `src/behaviorsense/__init__.py` - the one probe every
    # notebook makes - down the deep-search path it was written to avoid.
    # `weights/` is additionally tried for a bare filename, the staged weights layout.
    prefixes = ("", "EmotionSense-Extended/", "kaggle/working/")
    mids = ("",) if "/" in tail else ("", "weights/")
    for t in [p + m + tail for p in prefixes for m in mids]:
        hits = [s / t for s in _SLUGS if (s / t).is_file()]
        if hits:
            return hits[0]
    hits = sorted(INPUT.glob(f"**/{tail}"))   # last resort: unusual layout, slow, once
    if hits:
        print(f"  {what:<9} found only by deep search ({tail}) - layout is unusual")
        return hits[0]
    if required:
        raise AssertionError(
            f"{what}: nothing matches {tail!r} in any mount. Attached: {ATTACHED}")
    print(f"  {what:<9} ABSENT (optional)")
    return None

def find_asset(pattern, what, required=True):
    # Callers pass a `**/...` pattern. The leading `**/` is stripped and the mount-indexed
    # search runs first, so every existing call site gets the speed-up unchanged.
    tail = pattern[3:] if pattern.startswith("**/") else pattern
    return find_fast(tail, what, required=required)

def find_dir(subdir, pattern, roots=None):
    # "Which mount holds the most files matching this pattern in this subdirectory?" - one
    # scandir of ONE named directory per mount, never a recursive walk. Used for corpora
    # (Toyota's mp4/ and Videos_mp4/) where the answer is a directory, not a file.
    from fnmatch import fnmatch
    import os
    best, best_n = None, 0
    for root in (roots if roots is not None else _SLUGS):
        base = root / subdir if subdir else root
        if not base.is_dir():
            continue
        n = sum(1 for e in os.scandir(base) if e.is_file() and fnmatch(e.name, pattern))
        if n > best_n:
            best, best_n = base, n
    return best, best_n

def find_charades_csv():
    # The charades-480p dataset nests the CSV one level down (its root holds
    # Charades_annotations/ and Charades_v1_480/), and a `**` glob for it walks the code
    # dataset's MSMT17 copy - minutes for one file. Probe the known shapes per mount;
    # only a genuinely unknown layout falls through to the deep search.
    for slug in _SLUGS:
        if "charades" not in slug.name.lower():
            continue
        for base in (slug, slug / "Charades_annotations", slug / "Charades_v1_480",
                     slug / "kaggle" / "working"):
            p = base / "Charades_v1_train.csv"
            if p.is_file():
                return [p]
    return sorted(INPUT.glob("**/Charades_v1_train.csv"))

def find_wheel_dir():
    # "The directory containing *.whl" is not specific enough: /kaggle/input also holds
    # attached COMPETITIONS, and at least one (arc-prize-2026) ships its own wheels. The
    # first sorted hit was that competition's, and the offline install then failed on a
    # cache that simply does not contain torch. Score candidate directories by how many
    # of OUR packages they hold and take the best.
    MARKERS = {"torch", "rtmlib", "onnxruntime-gpu", "nvidia-cudnn-cu12", "triton"}
    dirs = {}
    for slug in _SLUGS:
        for wdir in (slug / "wheels", slug / "kaggle" / "working" / "wheels"):
            if not wdir.is_dir():
                continue
            for w in wdir.glob("*.whl"):
                dirs.setdefault(wdir, set()).add(
                    w.name.split("-")[0].lower().replace("_", "-"))
        if dirs:
            break
    if not dirs:
        for w in INPUT.glob("**/*.whl"):        # last resort
            dirs.setdefault(w.parent, set()).add(
                w.name.split("-")[0].lower().replace("_", "-"))
    if not dirs:
        raise AssertionError(f"no *.whl anywhere under /kaggle/input. Attached: {ATTACHED}")
    best, hits = max(dirs.items(), key=lambda kv: len(kv[1] & MARKERS))
    if not (hits & MARKERS):
        raise AssertionError(
            f"found {len(dirs)} wheel director(ies) but none holds any of {sorted(MARKERS)} "
            f"- the staged cache from notebook 00 is not attached. Candidates: "
            f"{[str(d) for d in dirs]}")
    return best

WHEELS  = find_wheel_dir()
WEIGHTS = find_asset("**/rtmo-l.onnx", "weights").parent
SRC     = find_asset("**/src/behaviorsense/__init__.py", "code").parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"

CONFIGS = CODE / "configs"

# sys.path belongs HERE, in the cell that resolves SRC, and unconditionally.
#
# It used to live at the end of the wheel-install cell. Notebook 04 put it outside that
# cell's `if not sm120_ok()` branch and worked; notebook 03 never had it at all and worked
# anyway, because every heavy step there is a subprocess launched with PYTHONPATH set. Then
# `is_real_artifact` was added to notebook 03's shard resolver - the first in-process import
# of `behaviorsense` in that notebook - and the next run died at cell 4 with
# `ModuleNotFoundError: No module named 'behaviorsense'`, six minutes in, one cell after
# PREFLIGHT PASSED. A path set up as a side effect of an unrelated, conditional cell is a
# dependency nobody can see.
import sys
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(SRC))

for _label, _path in (("wheels", WHEELS), ("weights", WEIGHTS), ("code", CODE)):
    print(f"  {_label:<8} {_path}")
print(f"  attached  {ATTACHED}")
print("NOTE: if you restart the kernel below, re-run from THIS cell - these names "
      "are what every later cell uses.")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_adl.py", "--tau-train",
     "notebook 03 passes --sampler/--tau-train; a snapshot predating them exits 2 from "
     "argparse, which notebook 03 reports as a failed stream rather than stale code"),
    ("src/behaviorsense/data/skeleton_dataset.py", "def load_subject_map",
     "video-id -> Charades actor-id remap for a person-disjoint P1 split (notebooks "
     "03/04). A stale snapshot silently reverts P1 to video-disjoint - same person in "
     "train and val - while printing numbers that look identical"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/ensemble.py", "def flip_windows",
     "test-time flip augmentation (notebook 04 levers cell). A stale snapshot would accept "
     "`clf.tta = True` as a new attribute and silently do no TTA, reporting the "
     "unaugmented number as if it were augmented"),
    ("src/behaviorsense/models/stgcnpp.py", "parents.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise. The "
     "token tracked the local name `parent` and broke when the function grew an explicit "
     "`parents` argument for Toyota's 15-node tree - a rename silently disarming a staleness "
     "guard is exactly what this list exists to catch, so it caught itself"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def force_greedy",
     "pins BOTH decoding arms to greedy (notebook 04). Without it the constrained arm "
     "inherits Qwen's generation_config (do_sample=True, temperature=0.7) while the free "
     "arm is greedy, so the comparison measures temperature instead of grammar - the "
     "constrained rate moved 8.2% -> 10.3% between two runs of identical code"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def _chat_text",
     "both arms send the SAME templated text (notebook 04). The constrained path used to "
     "hand outlines the raw prompt, so one arm got a Qwen chat turn and the other a naked "
     "instruction block - a second confound on top of the sampling one"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/eval/activity_eval.py", "def logit_adjust",
     "notebook 04's P1 cell imports MIN_SUPPORT and scores() from here, so a stale "
     "snapshot fails with ImportError at cell 3; also carries the post-hoc accuracy "
     "levers scripts/rescore_p1.py replays off the saved val logits"),
    ("src/behaviorsense/video.py", "def child_env",
     "notebook 05's /video endpoint decodes uploads in a CHILD process (ffmpeg raises SIGSEGV "
     "on malformed streams and a signal is not catchable, so without the boundary one bad "
     "upload kills the kernel, the tunnel and the demo together). `child_env` is what puts "
     "behaviorsense on that child's PYTHONPATH - sys.path does not cross a process boundary, "
     "and a snapshot without it 422s EVERY upload with \"No module named 'behaviorsense'\""),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
# Torch: use the preinstalled build if it supports sm_120, else install from the staged
# wheels. Deciding at runtime is the whole point of staging both.
import subprocess, sys
def sm120_ok() -> bool:
    try:
        import torch
        return torch.cuda.is_available() and "sm_120" in torch.cuda.get_arch_list()
    except Exception:
        return False
if not sm120_ok():
    print("preinstalled torch unusable on sm_120 - installing staged cu128 build")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index",
         "--find-links", str(WHEELS),
         "torch", "numpy", "pydantic", "PyYAML", "Pillow"],
        check=True)
    print("RESTART the kernel now (Run -> Restart) so the new torch is imported, "
          "then run from the next cell.")
else:
    import torch
    print(f"torch {torch.__version__} already supports sm_120")

In [ ]:
# Preflight: GPU arch, bf16 matmul, one ST-GCN++ fwd/bwd, every staged asset loads.
# Nonzero exit here means DO NOT start training - each listed problem surfaces late
# otherwise, after preprocessing time is already spent.
#
# --profile training scopes WHICH assets block. This notebook trains ST-GCN++ from
# pre-extracted .npz shards and loads no staged checkpoint, so the OSNet ReID weights are
# reported but do not fail it: they are Agent 1 serving-time, and their operating point
# (tau=0.3546) is already fitted in results/reid_eval.md. Everything that would actually
# stop this session - wrong arch, no bf16, a corrupt file - still blocks.
import subprocess, sys, os
# INHERIT the environment and only add PYTHONPATH. A hand-built env looks tidy and is
# wrong: it drops the CUDA variables Kaggle sets (LD_LIBRARY_PATH, NVIDIA_VISIBLE_DEVICES,
# CUDA_MODULE_LOADING...), so torch.cuda.is_available() is False in the child while the
# parent kernel sees the GPU perfectly well. The result was a preflight that reported
# "CUDA is not available - training would silently run on CPU" one line after the cell
# above printed "torch 2.10.0+cu128 already supports sm_120" - a self-contradicting
# session abort, caused entirely by this dict. tests/test_notebooks.py hit the same trap
# on Windows (a minimal env strips SystemRoot and breaks Winsock in torch.distributed)
# and fixed it there; the notebook kept the bug.
r = subprocess.run(
    [sys.executable, str(SCRIPTS / "kaggle_smoke_test.py"),
     "--assets", str(WEIGHTS), "--profile", "training"],
    env={**os.environ, "PYTHONPATH": str(SRC)})
assert r.returncode == 0, "PREFLIGHT FAILED - fix before burning GPU hours"


In [ ]:
# Collect shards from every attached shard dataset. Training accepts many --shards
# paths, which is why there is NO merge/combine notebook: merging would only copy bytes
# into a third dataset and double the storage without changing a single trained weight.
#
# ADL and fall shards are the same file type, so only the mount distinguishes them; match
# on the dataset name, tolerating the behaviour/behavior spelling, then assert non-empty.
# Training on an empty set for eleven hours is the failure this prevents.
from behaviorsense.kaggle_artifacts import is_real_artifact

def shard_paths(*keywords):
    # Group EVERY .npz by which ancestor directory names the corpus - do not iterate the
    # top level of /kaggle/input. Attaching by URL nests the mount as
    # /kaggle/input/datasets/<owner>/<name>/..., so `INPUT.iterdir()` yields just
    # ['competitions', 'datasets'], neither of which contains "adl" or "fall": both lists
    # came back EMPTY with the shard datasets correctly attached. The resolver cell above
    # never had this bug because it globs with **/; this cell was still name-on-top-level.
    out = []
    groups = {}
    for q in INPUT.glob("**/*.npz"):
        # Skip synthetic fixtures. `train_fall.py --smoke` writes data/shards/_smoke_fall.npz,
        # the code dataset is an upload of the working tree, and "_smoke_fall" contains
        # "fall" - so the fixture was matched into the real fall corpus and the head was
        # partly fitted on 400 windows of generated data. The leading underscore is the
        # convention every smoke fixture in this repo uses.
        if not is_real_artifact(q):
            print(f"  skipping {q.name} (fixture or code-checkout leftover)")
            continue
        rel = [part.lower().replace("behaviour", "behavior") for part in q.parts]
        owner = next((part for part in rel if any(k in part for k in keywords)), None)
        if owner:
            groups.setdefault(owner, []).append(str(q))
    for owner in sorted(groups):
        hits = sorted(groups[owner])
        print(f"  {owner}: {len(hits)} .npz")
        out += hits
    return sorted(out)

print("ADL mounts:")
ADL = shard_paths("adl")
print("fall mounts:")
FALL = shard_paths("fall")
print(f"ADL shards: {len(ADL)}, fall shards: {len(FALL)}")
assert ADL, f"no ADL shards found. Attached: {ATTACHED}"
assert FALL, f"no fall shards found. Attached: {ATTACHED}"

# The two sets must be DISJOINT. They are matched by mount name, so the only way a file
# appears in both is a dataset that actually holds the other corpus - e.g. Save Version
# publishing a working directory that still had the carried-forward ADL shards in it.
# That is not cosmetic: the fall head would count the same windows as positives and
# negatives, and P2's leave-one-dataset-out split would leak across folds. Equal counts
# in both lists is the symptom that prompted this check.
_dup = sorted(set(pathlib.Path(a).name for a in ADL)
              & set(pathlib.Path(f).name for f in FALL))
assert not _dup, (
    f"{len(_dup)} filename(s) appear in BOTH the ADL and fall shard sets: {_dup[:6]}"
    f"\n  One of those datasets contains the other corpus. Check the per-mount counts"
    f"\n  above, then re-publish the offending dataset from a clean /kaggle/working.")

# Report what the fall head will actually see. The fall corpora alone are ~82% positive;
# with the ADL windows as negatives it should land near 1-2%. A number outside that range
# means the mix is wrong, and every AUPRC/threshold downstream would be an artefact of it.
import numpy as np
_pos = _tot = 0
for _p in FALL + ADL:
    with np.load(_p) as _z:
        _lab = _z["labels"]
    _pos += int(np.isin(_lab, (7, 8)).sum()); _tot += int(_lab.size)
print(f"fall-head training mix: {_tot:,} windows, {_pos:,} positive "
      f"({_pos / max(_tot, 1):.2%})")
assert _tot > 0, "shards contain no windows at all"


In [ ]:
# Carry forward checkpoints for resume. Inputs are read-only; checkpoints get
# overwritten during training, so they must live in /kaggle/working.
import pathlib, shutil
RUNS = pathlib.Path("/kaggle/working/runs")
RUNS.mkdir(parents=True, exist_ok=True)
# Locate a previous run by the checkpoint itself. Same reasoning as the shard
# carry-forward in notebook 01: an unfound checkpoint must not read as "fresh",
# because resuming from nothing silently restarts an 11-hour run from epoch 0.
#
# But "any last.pt under /kaggle/input" is too generous, and it cost a session. The code
# dataset is an upload of the working tree, so it carried the LOCAL runs/ directory -
# CPU smoke checkpoints from `train_adl.py --smoke` (epochs=25, smoke=True). Those were
# copied in and resumed as though they were prior GPU work, which restarts an 80-epoch
# run on a 25-epoch cosine schedule. Nothing in the loss curve would say so.
#
# So every candidate is OPENED and checked before it is trusted. A checkpoint qualifies
# only if its own recorded args match the run this notebook is about to launch.
import torch
# Epoch budgets differ per head, so one WANT_EPOCHS is wrong: the ADL streams run 80 and
# the fall head runs 60. A single value rejected every legitimate fall checkpoint with
# "epochs=60, this run wants 80" - a false alarm on exactly the resume path this cell
# exists to serve. Keyed by the run directory name that the training cells write to.
# Budgets live HERE and the training cells read them, so the two cannot drift. They did:
# the ADL budget was cut 80 -> 30 after measuring that every stream peaked at epoch 8-11,
# and this guard was left at 80. It would have accepted the old 80-epoch checkpoints,
# then train_adl.py's own resume guard would have refused them ("--epochs was 80 but is
# 30 now") and killed all four streams at startup.
EPOCH_BUDGET = {"adl": 30, "fall": 60}
PATIENCE = {"adl": 8, "fall": 15}
WANT_EPOCHS = EPOCH_BUDGET

# Split identity for the ADL streams. The shards store VIDEO ids; Charades_v1_train.csv
# maps them to its 267 real actors, making the split person-disjoint. A checkpoint trained
# under one identity CANNOT resume under the other: the optimiser saw different windows, so
# carrying it forward would smuggle the old split's leakage into the new protocol. The
# acceptance check below therefore compares subject_map presence, not just epochs.
SUBJECT_MAP = find_charades_csv()
SUBJECT_MAP = str(SUBJECT_MAP[0]) if SUBJECT_MAP else None
print("ADL split identity:", "actor-id (Charades subjects)" if SUBJECT_MAP
      else "video-id proxy - attach charades-480p annotations for a person-disjoint P1")

def expected_epochs(run_name):
    # runs/adl_joint, runs/adl_bone_motion, ... -> "adl";  runs/fall -> "fall"
    return WANT_EPOCHS.get(run_name.split("_", 1)[0])

rejected = {}

def acceptable(ck_path):
    try:
        meta = torch.load(ck_path, map_location="cpu", weights_only=False)
    except Exception as exc:
        rejected[str(ck_path)] = f"unreadable ({type(exc).__name__})"
        return False
    args = meta.get("args") or {}
    if args.get("smoke") or args.get("smoke_shard"):
        rejected[str(ck_path)] = "smoke run (synthetic fixture, not real training)"
        return False
    want = expected_epochs(ck_path.parent.name)
    if want is None:
        rejected[str(ck_path)] = (f"unknown run directory {ck_path.parent.name!r} - this "
                                  "notebook only writes runs/adl_* and runs/fall")
        return False
    # A checkpoint trained for a different total changes the LR schedule on resume, and
    # train_adl.py refuses it anyway - better to skip it here than fail four streams in.
    if args.get("epochs") not in (None, want):
        rejected[str(ck_path)] = f"epochs={args.get('epochs')}, this run wants {want}"
        return False
    # Split-identity match, ADL runs only (the fall corpora already use real people).
    # bool-compare rather than path-compare: the same CSV mounts at a different absolute
    # path per session, and what changes the split is WHETHER it was used, not where from.
    if ck_path.parent.name.startswith("adl"):
        had_map = bool(args.get("subject_map"))
        if had_map != bool(SUBJECT_MAP):
            rejected[str(ck_path)] = (
                f"split identity mismatch: checkpoint was trained "
                f"{'actor-disjoint' if had_map else 'video-disjoint'}, this run is "
                f"{'actor-disjoint' if SUBJECT_MAP else 'video-disjoint'} - resuming would "
                "carry the other split's train/val boundary into this protocol")
            return False
    return True

ckpts = [c for c in sorted(INPUT.glob("**/last.pt")) if acceptable(c)]
if rejected:
    print(f"ignoring {len(rejected)} checkpoint(s) that are not resumable here:")
    for q, why in sorted(rejected.items()):
        print(f"  {pathlib.Path(q).parent.name:<14} {why}")
if ckpts:
    # Copy the ACCEPTED run directories one at a time. Copying their common parent would
    # drag rejected siblings along - the smoke checkpoints sat beside real ones, so a
    # parent-level copytree reinstates exactly what acceptable() just refused.
    for ck_path in ckpts:
        shutil.copytree(ck_path.parent, RUNS / ck_path.parent.name, dirs_exist_ok=True)
    found = sorted(q.parent.name for q in RUNS.glob("*/last.pt"))
    assert found, f"copied {len(ckpts)} checkpoint(s) but none landed in {RUNS}"
    print(f"resuming: {found}")
else:
    print("fresh training - no resumable last.pt under any attached dataset")

In [ ]:
# Train the 4 ADL streams. 96 GB VRAM fits all four CONCURRENTLY (each peaks at a few
# GB at batch 512 bf16) - wall-clock becomes the slowest stream instead of the sum.
# Set PARALLEL=False to serialise if anything OOMs.
import os, subprocess, sys, pathlib
PARALLEL = True
# 80 -> 30. Measured, not guessed: on the real Charades shards all four streams peaked
# at epoch 8-11 and then LOST ~27% mean-class-accuracy over the remaining 70 epochs
# (adl_bone 0.184@ep11 -> 0.137@ep79). 80 came from ST-GCN++'s NTU-60 recipe, which has
# ~10x more labelled windows per class. ~2.4 of the 2.7 GPU-hours went into memorising.
# --patience 8 ends each stream once it stops improving; best.pt already holds the peak,
# so this cannot cost a result - simulated against the real curve it stops at ep19.
EPOCHS, BATCH, SEED = str(EPOCH_BUDGET["adl"]), "512", "0"
STREAMS = ["joint", "bone", "joint_motion", "bone_motion"]
env = {**os.environ, "PYTHONPATH": str(SRC)}

# Class-imbalance strategy. mean-class accuracy is the metric of record and the label map
# leaves other_idle at 39.4%, so the objective matters. Two options, and they are mutually
# exclusive - train_adl.py refuses the combination because stacking them corrects the
# imbalance twice:
#   ("balanced", 0.0)  effective-number sampling, plain CE. What P1's 0.189 came from.
#   ("natural", 1.0)   natural batches, logit-adjusted loss (Menon et al. 2021).
# Effective-number weighting saturates by design: at 65,000 vs 69 windows it removes ~145x
# of a ~942x ratio, so residual imbalance survives it. Try the post-hoc adjustment on the
# EXISTING checkpoints first - scripts/rescore_p1.py does it on CPU in seconds off the saved
# val logits - and only spend a training run here if that is not enough.
SAMPLER, TAU_TRAIN = "balanced", 0.0

def launch(stream):
    out = f"/kaggle/working/runs/adl_{stream}"
    cmd = [sys.executable, str(SCRIPTS / "train_adl.py"),
           "--shards", *ADL, "--stream", stream, "--epochs", EPOCHS,
           "--batch-size", BATCH, "--device", "cuda", "--seed", SEED,
           "--patience", str(PATIENCE["adl"]), "--sampler", SAMPLER,
           "--tau-train", str(TAU_TRAIN), "--out", out]
    if SUBJECT_MAP:
        cmd += ["--subject-map", SUBJECT_MAP]   # person-disjoint split (see resolver cell)
    if pathlib.Path(out, "last.pt").exists():
        cmd += ["--resume", f"{out}/last.pt"]
    log = open(f"/kaggle/working/train_{stream}.log", "a")
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

if PARALLEL:
    procs = {s: launch(s) for s in STREAMS}
    RC = {s: p.wait() for s, p in procs.items()}
else:
    RC = {s: launch(s).wait() for s in STREAMS}
for s, rc in RC.items():
    print(s, "->", "OK" if rc == 0 else f"FAILED (see train_{s}.log)")
TRAINED_OK = sum(rc == 0 for rc in RC.values())

In [ ]:
# Fall head (focal loss, operating point fitted at a false-alarm budget - never 0.5).
#
# BOTH shard sets, not just the fall corpora. train_fall.py derives its binary target
# from labels 7/8, so every ADL window is simply a negative - and the fall shards alone
# are 81.6% POSITIVE (2501 fall / 563 non-fall as extracted). Two things break if the
# ADL shards are withheld:
#
#   1. The operating point stops meaning anything. fit_operating_point expresses the
#      budget as false alarms per HOUR of monitored video, and 563 negative windows is
#      0.31 h - 0.08 h after the val split. A budget of "1 FA/hour" then permits 0.08 of
#      one alarm, so the fitted threshold is whatever admits zero, which is an artefact
#      of test-set size rather than a deployment choice. Measured on a synthetic detector
#      of fixed quality: sensitivity 0.65 at 141 negatives vs 0.37 at a realistic mix,
#      AUPRC 0.993 vs 0.790. The same model, three different stories.
#   2. AUPRC is inflated for the same reason - it is the honest summary only when the
#      positive rate resembles deployment, and 81.6% positive resembles nothing.
#
# The ADL shards contribute ~165k negatives, which puts the positive rate near 1.5% -
# still optimistic against a real home, but the right side of realistic.
import subprocess, sys, os, pathlib
out = "/kaggle/working/runs/fall"
FALL_TRAIN = [*FALL, *ADL]
assert ADL, "ADL shards absent - the fall head would train on an 82% positive rate"
cmd = [sys.executable, str(SCRIPTS / "train_fall.py"),
       "--shards", *FALL_TRAIN, "--epochs", str(EPOCH_BUDGET["fall"]),
       "--batch-size", "256",
       # 60 epochs KEPT: unlike the ADL heads, the measured fall run peaked at ep56 of
       # 60 - still climbing near the end. Patience is wider for the same reason.
       "--device", "cuda", "--seed", "0", "--patience", str(PATIENCE["fall"]), "--out", out]
if pathlib.Path(out, "last.pt").exists():
    cmd += ["--resume", f"{out}/last.pt"]
r = subprocess.run(cmd, env={**os.environ, "PYTHONPATH": str(SRC)})
FALL_OK = r.returncode == 0
print("fall head:", "OK" if FALL_OK else "FAILED")

In [ ]:
# Session summary from each run's history.json - what to paste into the log book.
#
# Look in BOTH places, and REFUSE to print nothing. /kaggle/working is wiped between
# sessions, so re-running this cell in a fresh session finds an empty directory - the
# glob then yields zero iterations, the loop body never executes, and the cell ends by
# printing "Save Version" as though all were well. That is the same silent-empty-iteration
# failure this project has already fixed in notebooks 00, 01 and 02; it survived here.
#
# A completed batch run keeps its /kaggle/working as the version output, so attach that
# output (behaviorsense-runs) and the files are found under /kaggle/input instead.
import json, pathlib
def _real_runs(paths):
    # The code dataset is an upload of the working tree, so it carries the LOCAL runs/
    # directory - CPU smoke checkpoints (`--smoke`, 25 epochs) sitting beside the real
    # ones. The carry-forward cell already refuses to RESUME those; the summary was still
    # reading them and reporting a 25-epoch toy run as a result. Same exclusion, applied
    # to reporting: anything under a mounted code checkout, or named for a fixture.
    out = []
    for q in paths:
        parts = {part.lower() for part in q.parts}
        if "smoke" in q.parent.name.lower():
            continue
        if any("behaviorsense-code" in part or "emotionsense" in part for part in parts):
            continue
        out.append(q)
    return out

HISTS = sorted(pathlib.Path("/kaggle/working/runs").glob("*/history.json"))
WHERE = "/kaggle/working/runs"
if not HISTS:
    HISTS = _real_runs(sorted(pathlib.Path("/kaggle/input").glob("**/runs/*/history.json")))
    WHERE = "attached dataset"
if not HISTS:
    HISTS = _real_runs(sorted(pathlib.Path("/kaggle/input").glob("**/history.json")))
assert HISTS, (
    "no history.json anywhere. /kaggle/working is empty in a fresh session - the training "
    "outputs live in the completed run's version output. Attach that dataset "
    "(behaviorsense-runs) as an input, or re-run this notebook from the top. "
    "Searched: /kaggle/working/runs, then /kaggle/input/**")
print(f"reading {len(HISTS)} run(s) from {WHERE}")
for hist in HISTS:
    h = json.loads(hist.read_text())
    if not h:
        continue
    last = h[-1]
    # The key is `mean_class_acc`, not `mca`. Looking for the wrong name made every ADL
    # stream report `best=0.000` via max()'s default - the checkpoints were fine, the
    # summary was lying. A "best" that is exactly 0.000 for four independent runs is the
    # tell: real training noise never lands on precisely zero.
    # Identify the run by the metrics it RECORDED, not by its directory name. The name
    # heuristic broke on `smoke_fall` - a fall run whose name does not start with "fall" -
    # and asserted for `mean_class_acc` against a history that only has AUPRC. What a run
    # is, is what it measured.
    metric = ("auprc" if "auprc" in last else
              "mean_class_acc" if "mean_class_acc" in last else None)
    if metric is None:
        print(f"  {hist.parent.name:<18} SKIPPED - no recognised metric "
              f"(keys: {sorted(last)[:6]})")
        continue
    shown = [k for k in ("mean_class_acc", "macro_f1", "macro_f1_supported", "top1",
                         "auprc", "auroc", "sensitivity") if k in last]
    best = max(e[metric] for e in h if metric in e)
    best_ep = next(e.get("epoch", i) for i, e in enumerate(h)
                   if e.get(metric) == best)
    print(f"{hist.parent.name:<18} epoch {last.get('epoch', len(h)-1):>3}  "
          + "  ".join(f"{k}={last[k]:.3f}" for k in shown)
          + f"  | best {metric}={best:.3f} @ep{best_ep}")

# Per-class breakdown for the ADL streams. macro-F1 alone cannot distinguish "the task is
# hard" from "the pipeline is broken", and those need opposite responses. The vector is
# already in history.json (per_class_f1 / per_class_support) - it just was never printed.
#
# Read it like this:
#   a broad low-but-nonzero spread   -> genuinely hard (Charades is untrimmed and
#                                       multi-label; a 2 s window often contains no
#                                       evidence of the labelled action at all)
#   a few classes at ~0, rest fine   -> label-map starvation, already documented
#   everything ~0 except other_idle  -> collapse; the signal is being destroyed upstream
CLASS_NAMES = ["walking", "standing", "sitting", "lying_down", "standing_up",
               "sitting_down", "bending_reaching", "falling", "fallen_on_ground",
               "eating", "drinking", "cooking_food_prep", "taking_medication",
               "watching_tv", "reading", "using_phone", "cleaning_housework",
               "personal_hygiene", "interacting_with_person", "other_idle"]
for hist in [q for q in HISTS if q.parent.name.startswith("adl")]:
    h = json.loads(hist.read_text())
    best = max(h, key=lambda e: e.get("mean_class_acc", 0))
    f1s, sup = best.get("per_class_f1"), best.get("per_class_support")
    if not f1s:
        continue
    print()
    print(f"{hist.parent.name} - per-class F1 at best epoch {best.get('epoch')}:")
    rows = sorted(((f1s[i], sup[i], CLASS_NAMES[i]) for i in range(len(f1s)) if sup[i]),
                  reverse=True)
    for f1, n, name in rows:
        bar = "#" * int(f1 * 40)
        print(f"  {name:<24} n={n:>6}  f1={f1:.3f}  {bar}")
    dead = [name for f1, n, name in rows if f1 < 0.02]
    print(f"  -> {len(rows) - len(dead)}/{len(rows)} classes above F1 0.02; "
          f"{len(dead)} at ~0: {dead[:6]}")
    break            # one stream is enough to diagnose; they share the same data

print()
# Do NOT tell the user to publish a session that trained nothing. This cell used to print
# "Save Version -> create/update dataset behaviorsense-runs" unconditionally, and it printed
# it after a session where all five trainers had crashed on startup: /kaggle/working/runs
# held byte-identical carry-forward copies, and the only new files were four crash
# tracebacks in train_*.log - which a new dataset version would have published OVER the real
# training logs. Advice that is wrong in exactly the situation where it is most likely to be
# followed is worse than no advice.
_ok = globals().get("TRAINED_OK", 0) + int(globals().get("FALL_OK", False))
if _ok:
    print(f"{_ok} run(s) advanced this session.")
    print("Save Version -> create/update dataset behaviorsense-runs from /kaggle/working/runs")
    print("Timed out mid-training? Save Version anyway - resume continues bit-exactly next "
          "session.")
else:
    print("NOTHING TRAINED this session - every run failed to start or had already"
          " finished.")
    print("Do NOT publish a new behaviorsense-runs version: /kaggle/working/runs holds"
          " carry-forward copies")
    print("of the checkpoints you already have, and train_*.log here contains only this"
          " session's errors,")
    print("which would replace the real training logs in the existing dataset.")
    print("Fix the cause above, or go straight to notebook 04 - best.pt already holds every"
          " peak.")